In [ ]:
import numpy as np
import torch
import torch.nn as nn

from dlfs_autograd.layers import DenseLayer
from dlfs_autograd.tensor.base import Parameter
from .helpers import compare_arrays

In [ ]:
use_autograd = True

np.random.seed(0)
torch.manual_seed(0)

B = 15
features_in=23
features_out = 35

x = np.random.normal(size=(B, features_in))
print(x.shape)

In [ ]:
linear = nn.Linear(features_in, features_out)
with torch.no_grad():
    linear.bias.zero_()

torch_x = torch.Tensor(x)

torch_out = linear(torch_x)
print(f'Torch linear out: {torch_out.shape}')

In [ ]:
torch_weight = linear.weight.detach().numpy()
print(f'Torch linear weights: {torch_weight.shape}')

In [ ]:
linear_dlfs = DenseLayer(features_in, features_out)
print(f'DLFS linear weights: {linear_dlfs.weights.shape}')

linear_dlfs.weights = torch_weight.T

if use_autograd:
    linear_dlfs.weights = Parameter(
        torch_weight.T
    )

In [ ]:
linear_dlfs.forward(x)

dlfs_out = linear_dlfs.output

#if use_autograd:
#    dlfs_out = linear_dlfs.output.data

print(f'DLFS linear out: {dlfs_out.shape}')

# Linear output comparison

In [ ]:
compare_arrays(torch_out.detach().numpy(), dlfs_out.data)

# Linear gradient comparison

In [ ]:
torch_sum = torch.sum(torch_out)
print(torch_sum)
torch_sum.backward()

In [ ]:
dlfs_sum = dlfs_out.sum()
print(dlfs_sum)
dlfs_sum.backward()

In [ ]:
torch_grad = linear.weight.grad.detach().cpu().numpy()
dlfs_grad = linear_dlfs.weights.grad

compare_arrays(torch_grad.T, dlfs_grad)